# Notebook 04 — Camada Gold (Esquema Estrela)

**MVP de Engenharia de Dados** · PUC-Rio · Sprint 3

---

### Objetivo

Modelar os dados tipados em um **esquema estrela**, estrutura otimizada para consultas analíticas, e validar a corretude do pipeline.

### Entrada e saída

| | |
|---|---|
| **Entrada** | `workspace.silver.balanco_energia` |
| **Saída** | 3 dimensões + 2 tabelas fato + validação em `workspace.gold` |

### O modelo

| Tabela | Grão | Registros |
|---|---|---|
| `dim_tempo` | hora | 61.368 |
| `dim_subsistema` | subsistema | 5 |
| `dim_fonte` | fonte de geração | 4 |
| `fato_geracao` | hora × subsistema × fonte | 981.888 |
| `fato_carga` | hora × subsistema | 245.472 |

### Por que duas tabelas fato

Porque elas têm **grãos diferentes**: a geração é medida por fonte, a carga não.

Forçar as duas em uma única tabela obrigaria a repetir o valor de carga em cada linha de fonte — e qualquer `SUM(carga)` passaria a contar o mesmo valor quatro vezes. Separar os fatos e compartilhar as mesmas dimensões (o que se chama **dimensões conformadas**) permite consultar as duas em conjunto sem esse risco.

## 1. Dimensões

**`dim_subsistema`** é extraída dos próprios dados. Mantém o SIN, marcado por `flag_agregado` — as tabelas fato é que vão excluí-lo.

**`dim_fonte`** não vem dos dados: é construída a partir de **conhecimento de domínio** do setor elétrico. Os indicadores `flag_renovavel` e `flag_intermitente` são classificações que nenhuma coluna da origem fornece, mas que as perguntas de negócio exigem. A coluna `col_origem` registra de qual campo da Silver cada fonte veio — linhagem dentro da própria dimensão.

**`dim_tempo`** enriquece o instante com ano, mês, dia, hora, trimestre, dia da semana, indicador de fim de semana e estação do ano (calculada para o hemisfério sul). Calcular isso uma vez, na dimensão, evita repetir a mesma lógica em cada consulta analítica.

**Chaves substitutas (*surrogate keys*).** Cada dimensão recebe um identificador próprio, independente da chave natural da origem. Se amanhã o ONS renomear um subsistema, as tabelas fato continuam íntegras.

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

spark.conf.set("spark.sql.session.timeZone", "UTC")
silver = spark.table("workspace.silver.balanco_energia")

# --- dim_subsistema ---
dim_subsistema = (
    silver.select("id_subsistema", "nom_subsistema", "flag_agregado").distinct()
          .withColumn("sk_subsistema",
                      F.row_number().over(Window.orderBy("id_subsistema")))
          .select("sk_subsistema", "id_subsistema", "nom_subsistema", "flag_agregado")
)

# --- dim_fonte (construída a partir do conhecimento de domínio) ---
dim_fonte = spark.createDataFrame(
    [(1, "Hidráulica",   "val_gerhidraulica", True,  False),
     (2, "Térmica",      "val_gertermica",    False, False),
     (3, "Eólica",       "val_gereolica",     True,  True),
     (4, "Fotovoltaica", "val_gersolar",      True,  True)],
    "sk_fonte int, nom_fonte string, col_origem string, "
    "flag_renovavel boolean, flag_intermitente boolean"
)

# --- dim_tempo ---
md = F.month("din_instante") * 100 + F.dayofmonth("din_instante")

dim_tempo = (
    silver.select("din_instante").distinct()
      .withColumn("sk_tempo",   F.date_format("din_instante", "yyyyMMddHH").cast("long"))
      .withColumn("data",       F.to_date("din_instante"))
      .withColumn("ano",        F.year("din_instante"))
      .withColumn("mes",        F.month("din_instante"))
      .withColumn("dia",        F.dayofmonth("din_instante"))
      .withColumn("hora",       F.hour("din_instante"))
      .withColumn("trimestre",  F.quarter("din_instante"))
      .withColumn("dia_semana", F.dayofweek("din_instante"))
      .withColumn("flag_fim_semana", F.dayofweek("din_instante").isin(1, 7))
      .withColumn("estacao",
          F.when((md >= 1221) | (md <= 320), "Verão")
           .when((md >= 321) & (md <= 620), "Outono")
           .when((md >= 621) & (md <= 922), "Inverno")
           .otherwise("Primavera"))
)

for nome, df_dim in [("dim_subsistema", dim_subsistema),
                     ("dim_fonte",      dim_fonte),
                     ("dim_tempo",      dim_tempo)]:
    (df_dim.write.mode("overwrite").option("overwriteSchema", "true")
           .saveAsTable(f"workspace.gold.{nome}"))
    print(f"{nome:16s} -> {df_dim.count():>8,} linhas")
    

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1163: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


dim_subsistema   ->        5 linhas
dim_fonte        ->        4 linhas
dim_tempo        ->   61,368 linhas


## 2. Tabelas fato

Duas operações estruturais acontecem aqui.

**Unpivot com `stack`.** Os dados da origem vêm em formato **largo**: quatro colunas de geração lado a lado. Uma tabela fato com dimensão de fonte precisa do formato **longo**: uma linha por fonte. O `stack(4, ...)` faz essa transposição, multiplicando por quatro o número de linhas — 245.472 registros viram 981.888.

**Exclusão do SIN.** O filtro `~flag_agregado` remove o total nacional. É aqui, e só aqui, que a separação acontece: a Silver preservou tudo, a Gold entrega apenas o dado detalhado.

Em seguida, os `join` com as dimensões substituem as chaves naturais pelas chaves substitutas.

In [0]:
# fato_geracao: grão = hora x subsistema x fonte
# O stack() faz o "unpivot": transforma 4 colunas de fonte em 4 linhas.
fato_geracao = (
    silver.filter(~F.col("flag_agregado"))          # exclui o total nacional
      .selectExpr(
          "id_subsistema", "din_instante",
          """stack(4,
               'Hidráulica',   val_gerhidraulica,
               'Térmica',      val_gertermica,
               'Eólica',       val_gereolica,
               'Fotovoltaica', val_gersolar
             ) as (nom_fonte, val_geracao_mwmed)""")
      .withColumn("sk_tempo", F.date_format("din_instante", "yyyyMMddHH").cast("long"))
      .join(dim_subsistema.select("id_subsistema", "sk_subsistema"), "id_subsistema")
      .join(dim_fonte.select("nom_fonte", "sk_fonte"), "nom_fonte")
      .select("sk_tempo", "sk_subsistema", "sk_fonte", "val_geracao_mwmed")
)

# fato_carga: grão = hora x subsistema
fato_carga = (
    silver.filter(~F.col("flag_agregado"))
      .withColumn("sk_tempo", F.date_format("din_instante", "yyyyMMddHH").cast("long"))
      .join(dim_subsistema.select("id_subsistema", "sk_subsistema"), "id_subsistema")
      .select("sk_tempo", "sk_subsistema",
              F.col("val_carga").alias("val_carga_mwmed"),
              F.col("val_intercambio").alias("val_intercambio_mwmed"))
)

for nome, df_fato in [("fato_geracao", fato_geracao), ("fato_carga", fato_carga)]:
    (df_fato.write.mode("overwrite").option("overwriteSchema", "true")
            .saveAsTable(f"workspace.gold.{nome}"))
    print(f"{nome:16s} -> {df_fato.count():>10,} linhas")

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1163: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


fato_geracao     ->    981,888 linhas
fato_carga       ->    245,472 linhas


## 3. Validação cruzada do pipeline

Aqui o registro SIN — aquele mesmo que causaria dupla contagem se fosse ignorado — é reaproveitado como **referência independente de corretude**.

O raciocínio: se a soma da geração dos quatro subsistemas bater com o total nacional publicado pelo ONS, então o unpivot não perdeu nem duplicou linhas, os filtros removeram exatamente o que deveriam, e os joins com as dimensões preservaram todos os registros.

É um teste de ponta a ponta usando um valor que o próprio pipeline não calculou.

**Nota sobre o resultado.** A diferença máxima esperada não é zero absoluto, e sim algo na casa de 0,003 MWmed. Números decimais em ponto flutuante acumulam um erro microscópico a cada soma, e aqui são quase um milhão de valores somados. Um zero exato seria mais suspeito que esse resíduo.

In [0]:
# A soma dos 4 subsistemas deve bater com a linha SIN do mesmo instante.
# Se bater, o unpivot e os filtros do pipeline estão corretos.
soma_4 = (spark.table("workspace.gold.fato_geracao")
            .groupBy("sk_tempo")
            .agg(F.sum("val_geracao_mwmed").alias("soma_subsistemas")))

sin = (silver.filter(F.col("flag_agregado"))
         .withColumn("sk_tempo", F.date_format("din_instante", "yyyyMMddHH").cast("long"))
         .select("sk_tempo",
                 (F.col("val_gerhidraulica") + F.col("val_gertermica")
                  + F.col("val_gereolica") + F.col("val_gersolar")).alias("total_sin")))

dif = (soma_4.join(sin, "sk_tempo")
         .withColumn("diferenca", F.abs(F.col("soma_subsistemas") - F.col("total_sin"))))

print(f"Instantes comparados      : {dif.count():,}")
print(f"Diferença máxima (MWmed)  : {dif.agg(F.max('diferenca')).collect()[0][0]:.4f}")
print(f"Instantes com dif > 1     : {dif.filter(F.col('diferenca') > 1).count():,}")

Instantes comparados      : 61,368
Diferença máxima (MWmed)  : 0.0034
Instantes com dif > 1     : 0
